## Задание 1

Для каждого покупателя получите таблицу со столбцами:
* preferred_payment - самый частый метод оплаты (при ничьей зафиксируйте правило, например lexicographically первый);
* total_spent - сумма всех трат;
* addons_spent - сумма на дополнительные услуги и аксессуары;
* orders_count - число заказов.
 
Отсортируйте по total_spent по убыванию и выведите топ-10.
 
Пограничные случаи:

* Покупатель с одним заказом - orders_count = 1, preferred_payment = единственный метод.
* Два метода оплаты с одинаковой частотой - выберите один по зафиксированному правилу (укажите его в комментарии к коду).
* Покупатель без доп. услуг/аксессуаров - addons_spent = 0.
* Пропуски в сумме или методе оплаты - обработайте (отфильтруйте или заполните) и кратко опишите решение в комментарии.
 
 
Задание 1. Код выводит топ-10 покупателей с указанием: preferred_payment, total_spent, addons_spent, orders_count.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Electronic_sales_Sep2023-Sep2024.csv', sep=',')
df.info()
display(df.head(2))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  object 
 3   Loyalty Member     20000 non-null  object 
 4   Product Type       20000 non-null  object 
 5   SKU                20000 non-null  object 
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  object 
 8   Payment Method     20000 non-null  object 
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  object 
 13  Shipping Type      20000 non-null  object 
 14  Add-ons Purchased  15132 non-null  object 
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09


In [3]:
# в исходных данных есть строка с пропуском в Gender, удаляем
df = df.dropna(subset=['Gender'])

In [4]:
print('Payments methods:', df['Payment Method'].unique(), sep='\n')
print('*' * 70)
print('Total Price:', df['Total Price'].describe(), sep='\n')
print('*' * 70)
print('Add-on Total', df['Add-on Total'].describe(), sep='\n')

Payments methods:
['Credit Card' 'Paypal' 'Cash' 'Debit Card' 'Bank Transfer' 'PayPal']
**********************************************************************
Total Price:
count    19999.000000
mean      3180.258715
std       2544.980615
min         20.750000
25%       1139.680000
50%       2534.490000
75%       4639.600000
max      11396.800000
Name: Total Price, dtype: float64
**********************************************************************
Add-on Total
count    19999.000000
mean        62.247960
std         58.058214
min          0.000000
25%          7.625000
50%         51.700000
75%         93.845000
max        292.770000
Name: Add-on Total, dtype: float64


In [5]:
# удаление разночтений в видах платежа
df['Payment Method'].replace('PayPal', 'Paypal', inplace=True)
df['Payment Method'].unique()

array(['Credit Card', 'Paypal', 'Cash', 'Debit Card', 'Bank Transfer'],
      dtype=object)

In [6]:
# фильтруем заверешенные заказы без учета отмененных ?
df = df.loc[df['Order Status'] == 'Completed'].copy()
df.shape[0]

13431

In [7]:
# проверка значений с пропусками по Adds-ons Purchased
print(df[df['Add-ons Purchased'].isna()]['Add-on Total'].unique())
print(df[df['Add-on Total'] == 0]['Add-ons Purchased'].unique())

[0.]
[nan]


In [8]:
# самый частый метод оплаты
payment_method = (
    df.groupby(['Customer ID', 'Payment Method'])
      .size()
      .reset_index(name='Method count')
)
payment_method.sort_values(
    ['Customer ID', 'Payment Method', 'Method count']
    , ascending=[True, True, False]
    , inplace=True)
payment_method.drop_duplicates(subset='Customer ID', keep='first', inplace=True)
payment_method.drop('Method count', axis=1, inplace=True)
payment_method.rename(columns={'Payment Method': 'preferred_payment'}, inplace=True)
payment_method.shape

(9465, 2)

In [20]:
# сумма всех трат, сумма на дополнительные услуги и аксессуары и число заказов.
df_pre_final = (
    df.groupby('Customer ID')
      .agg(
          total_spent=('Total Price', 'sum'),
          addons_spent=('Add-on Total', 'sum'),
          orders_count=('Customer ID', 'size')
      ).reset_index()
)
df_pre_final.shape

(9465, 4)

In [21]:
df_final = pd.merge(
    df_pre_final
    ,payment_method
    , how='inner')

display(
    df_final[[
        'Customer ID'
        ,'preferred_payment'
        ,'total_spent'
        ,'addons_spent'
        ,'orders_count'
    ]].sort_values('total_spent', ascending=False).head(10))

,Customer ID,preferred_payment,total_spent,addons_spent,orders_count
5048,11476,Credit Card,29937.93,198.42,4
7095,15399,Bank Transfer,29084.88,305.39,3
6177,13635,Bank Transfer,28093.28,388.11,4
7584,16357,Bank Transfer,27486.01,352.75,6
4960,11332,Bank Transfer,26725.65,240.24,4
5489,12294,Bank Transfer,26575.02,477.68,4
839,2556,Cash,26098.40,230.66,6
6124,13534,Bank Transfer,25893.60,367.61,5
6589,14436,Credit Card,24912.42,102.31,4
6266,13797,Bank Transfer,24870.42,78.83,3


## Задание 2

Постройте пайплайн на Pandas (без циклов по строкам):
 
 1. Очистка: типы дат, пропуски, аномалии (отрицательные суммы / нулевые цены) - кратко опишите, что отфильтровали.
 2. Метрики:
    • доход по методу доставки;
    • доход по типу продукта;
    • доход по доп. услугам: по месяцам и по кварталам;
    • cohort-анализ: когорта = месяц первого заказа покупателя; для каждой когорты - выручка и число покупателей в следующие 0, 1, 2, 3 месяца.
 3. Визуализация (минимум 3 графика): bar (доставка / тип продукта), line (доп. услуги по месяцам), heatmap когортной матрицы выручки.
 4. Выводы: несколько предложений по графикам. (?необязательно)
 
 Дополнительно: оформите шаги 1–2 как функции clean(df) → df и build_metrics(df) → dict[str, DataFrame].
 
 
 Пограничные случаи:

• Заказы с отрицательной или нулевой ценой - не должны попадать в доход после clean.
 • Покупатель с одним заказом - в когортной матрице заполнен только месяц 0.
 • Месяц без доп. услуг - в addons_by_month строка с 0 или отсутствие месяца (выберите и зафиксируйте поведение).
 • Пустой датасет после очистки → функции не падают, возвращают пустые таблицы.
 
 Задание 2. Код выводит метрики (clean / build_metrics) и отображает их минимум на трёх графиках.